# Dataset Integrity Audit

An independent check that the Tuktoyaktuk training data and its
train/validation split are not contaminated. **Deliberately does not import
or trust any other notebook's assertions** -- it recomputes the split from
the raw patch geometry and tests it from scratch, so a bug in the split
logic cannot hide behind its own passing assert.

Two things have already gone wrong in this project that no metric caught:

1. A random patch-level split leaked, because extracted patches overlap
   spatially. ZNCC 0.519 -> 0.2344 once fixed.
2. The ArcticDEM conditioning input was a constant (sea level over sea
   ice), so the DEM branch carried no information at all -- caught only by
   printing the input's within-patch variance and comparing it to the
   target's.

Both were invisible in loss curves, parameter counts, and metric
confidence intervals. This notebook checks the class of thing that hides
that way.

**Ten checks.** Each prints PASS / FAIL / NOTE independently, and the last
cell summarises. A NOTE is something to describe in the write-up rather
than a defect.

**CPU only.** Runs while the GPU is busy.

## Setup

In [1]:
import json
import math
import random
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import box
from shapely.strtree import STRtree

WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc'
DEM_DIR = WORKING_REPO / 'input_data' / 'dem_patches_tuk'
CB_LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_cambridge_extracted' / 'lidar_patches_cambridge'
CB_S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_cambridge_pcrtc'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'

# Split parameters, as used by pcrtc/09 and dem_unet/03
BLOCK_SIZE_M = 1024.0
BUFFER_M = 150.0
VAL_FRACTION = 0.15
SEED = 42

HASH_S1 = True   # set False to skip check 4 if it is slow

results = {}
def record(name, status, detail=''):
    results[name] = (status, detail)
    print(f'[{status}] {name}' + (f'  --  {detail}' if detail else ''))

print('LiDAR:', LIDAR_DIR)
print('S1:   ', S1_DIR)

LiDAR: /cs/student/project_msc/2025/aibh/jiayiche/input_data/lidar_patches_tuk_tessa
S1:    /cs/student/project_msc/2025/aibh/jiayiche/input_data/s1_patches_tuk_pcrtc


## Reproduce the split from raw geometry

Copied verbatim from `pcrtc/09` / `dem_unet/03`. If this does not reproduce
534 / 255 / 887, the split is not what the trained checkpoints used and
nothing downstream can be compared.

In [2]:
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)

bounds = {}
crs_seen = set()
for pid in paired_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        bounds[pid] = src.bounds
        crs_seen.add(str(src.crs))

def patch_centroid(pid):
    b = bounds[pid]
    return ((b.left + b.right) / 2.0, (b.bottom + b.top) / 2.0)

centroids = {pid: patch_centroid(pid) for pid in paired_ids}

def block_id_and_boundary_distance(cx, cy, block_size):
    bx, by = int(cx // block_size), int(cy // block_size)
    dx = min(cx - bx * block_size, (bx + 1) * block_size - cx)
    dy = min(cy - by * block_size, (by + 1) * block_size - cy)
    return (bx, by), min(dx, dy)

blocks, dropped_buffer = {}, []
for pid, (cx, cy) in centroids.items():
    bid, dist = block_id_and_boundary_distance(cx, cy, BLOCK_SIZE_M)
    if dist < BUFFER_M:
        dropped_buffer.append(pid)
        continue
    blocks.setdefault(bid, []).append(pid)

block_ids = list(blocks.keys())
random.Random(SEED).shuffle(block_ids)
target_val = int(len(paired_ids) * VAL_FRACTION)
val_ids, train_ids, running = [], [], 0
for bid in block_ids:
    if running < target_val:
        val_ids.extend(blocks[bid]); running += len(blocks[bid])
    else:
        train_ids.extend(blocks[bid])

print(f'paired={len(paired_ids)}  train={len(train_ids)}  val={len(val_ids)}  '
      f'dropped_buffer={len(dropped_buffer)}')
print(f'LiDAR CRS values present: {crs_seen}')

ok = (len(train_ids), len(val_ids), len(dropped_buffer)) == (534, 255, 887)
record('0. Split reproduces 534/255/887', 'PASS' if ok else 'FAIL',
       '' if ok else f'got {len(train_ids)}/{len(val_ids)}/{len(dropped_buffer)}')
record('0b. Single LiDAR CRS', 'PASS' if len(crs_seen) == 1 else 'FAIL', ', '.join(crs_seen))

paired=1676  train=534  val=255  dropped_buffer=887
LiDAR CRS values present: {'EPSG:32608'}
[PASS] 0. Split reproduces 534/255/887
[PASS] 0b. Single LiDAR CRS  --  EPSG:32608


## Check 1 -- do the extracted patches overlap each other at all?

The root cause of the original leak. If patches are disjoint this whole
class of problem cannot arise; if they overlap heavily, the block split is
load-bearing and must be exactly right.

In [3]:
all_boxes = [box(*bounds[p]) for p in paired_ids]
tree_all = STRtree(all_boxes)

overlap_counts = []
for i, g in enumerate(all_boxes):
    hits = [j for j in tree_all.query(g) if j != i]
    real = [j for j in hits if all_boxes[j].intersects(g) and not all_boxes[j].touches(g)]
    overlap_counts.append(len(real))

overlap_counts = np.array(overlap_counts)
frac_overlapping = float((overlap_counts > 0).mean())
print(f'Patches overlapping >=1 other patch: {frac_overlapping:.1%}')
print(f'Median overlapping neighbours per patch: {np.median(overlap_counts):.0f}   max: {overlap_counts.max()}')

record('1. Patch overlap characterised', 'NOTE',
       f'{frac_overlapping:.0%} of patches overlap a neighbour -- '
       f'a random split WOULD leak; the block split is load-bearing')

Patches overlapping >=1 other patch: 100.0%
Median overlapping neighbours per patch: 7   max: 8
[NOTE] 1. Patch overlap characterised  --  100% of patches overlap a neighbour -- a random split WOULD leak; the block split is load-bearing


## Check 2 -- geometric separation of train and validation

The decisive leakage test. Any val patch sharing area with a train patch is
direct contamination. Also reports the *distance* distribution: patches can
be non-overlapping yet close enough to be effectively the same terrain.

In [4]:
train_boxes = [box(*bounds[p]) for p in train_ids]
tree_train = STRtree(train_boxes)

n_overlap = 0
overlapping_pairs = []
min_dists = []
for pid in val_ids:
    g = box(*bounds[pid])
    hits = tree_train.query(g)
    real = [j for j in hits if train_boxes[j].intersects(g) and not train_boxes[j].touches(g)]
    if real:
        n_overlap += 1
        overlapping_pairs.append((pid, [train_ids[j] for j in real[:3]]))
    nearest_j = tree_train.nearest(g)
    min_dists.append(g.distance(train_boxes[nearest_j]))

min_dists = np.array(min_dists)
print(f'Validation patches overlapping a training patch: {n_overlap} / {len(val_ids)}')
print(f'Distance to nearest training patch (metres):')
print(f'   min {min_dists.min():.1f}   p5 {np.percentile(min_dists,5):.1f}   '
      f'median {np.median(min_dists):.1f}   max {min_dists.max():.1f}')
if overlapping_pairs:
    print('Examples:', overlapping_pairs[:5])

record('2. No train/val geometric overlap', 'PASS' if n_overlap == 0 else 'FAIL',
       f'{n_overlap} overlapping val patches')
close = int((min_dists < BUFFER_M).sum())
record('2b. Val patches within buffer distance of train',
       'PASS' if close == 0 else 'NOTE',
       f'{close} val patches lie closer than {BUFFER_M:.0f}m to a training patch '
       f'(min {min_dists.min():.1f}m)')

Validation patches overlapping a training patch: 0 / 255
Distance to nearest training patch (metres):
   min 128.0   p5 128.0   median 384.0   max 819.6
[PASS] 2. No train/val geometric overlap  --  0 overlapping val patches
[NOTE] 2b. Val patches within buffer distance of train  --  23 val patches lie closer than 150m to a training patch (min 128.0m)


## Check 3 -- exact duplicate LiDAR patches

Two patches with byte-identical target arrays are the same data under two
ids. Split across train and val, that is leakage with no geometric overlap
to reveal it.

In [5]:
def array_hash(arr):
    return hashlib.sha256(np.ascontiguousarray(arr).tobytes()).hexdigest()

lidar_hash, lidar_stats = {}, {}
for pid in paired_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        raw = src.read().astype(np.float32)
    tgt = raw[0]
    mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(tgt)
    lidar_hash[pid] = array_hash(np.nan_to_num(tgt))
    valid = tgt[mask]
    lidar_stats[pid] = {
        'valid_frac': float(mask.mean()),
        'std': float(valid.std()) if valid.size > 10 else 0.0,
        'n_valid': int(mask.sum()),
    }

groups = defaultdict(list)
for pid, h in lidar_hash.items():
    groups[h].append(pid)
dupes = {h: ids for h, ids in groups.items() if len(ids) > 1}

train_set, val_set = set(train_ids), set(val_ids)
cross_split_dupes = [ids for ids in dupes.values()
                     if any(i in train_set for i in ids) and any(i in val_set for i in ids)]

print(f'Distinct LiDAR arrays: {len(groups)} / {len(paired_ids)}')
print(f'Duplicate groups (any): {len(dupes)}')
print(f'Duplicate groups spanning train AND val: {len(cross_split_dupes)}')
if cross_split_dupes:
    print('Examples:', cross_split_dupes[:5])

record('3. No duplicate LiDAR across train/val',
       'PASS' if not cross_split_dupes else 'FAIL',
       f'{len(cross_split_dupes)} cross-split duplicate groups')
record('3b. Duplicate patches within the dataset',
       'PASS' if not dupes else 'NOTE',
       f'{len(dupes)} duplicate groups total')

Distinct LiDAR arrays: 1676 / 1676
Duplicate groups (any): 0
Duplicate groups spanning train AND val: 0
[PASS] 3. No duplicate LiDAR across train/val  --  0 cross-split duplicate groups
[PASS] 3b. Duplicate patches within the dataset  --  0 duplicate groups total


## Check 4 -- exact duplicate Sentinel-1 conditioning stacks

In [6]:
if HASH_S1:
    s1_hash = {}
    for pid in paired_ids:
        times = sorted((S1_DIR / f's1_patch_{pid}').glob('t*.tif'))[:3]
        h = hashlib.sha256()
        for t in times:
            with rasterio.open(t) as src:
                h.update(np.ascontiguousarray(np.nan_to_num(src.read()[:2].astype(np.float32))).tobytes())
        s1_hash[pid] = h.hexdigest()

    s1_groups = defaultdict(list)
    for pid, h in s1_hash.items():
        s1_groups[h].append(pid)
    s1_dupes = {h: ids for h, ids in s1_groups.items() if len(ids) > 1}
    s1_cross = [ids for ids in s1_dupes.values()
                if any(i in train_set for i in ids) and any(i in val_set for i in ids)]

    print(f'Distinct S1 stacks: {len(s1_groups)} / {len(paired_ids)}')
    print(f'Duplicate S1 groups spanning train AND val: {len(s1_cross)}')
    if s1_cross:
        print('Examples:', s1_cross[:5])
    record('4. No duplicate S1 stacks across train/val',
           'PASS' if not s1_cross else 'FAIL', f'{len(s1_cross)} cross-split groups')
else:
    record('4. S1 duplicate check', 'SKIP', 'HASH_S1 = False')

Distinct S1 stacks: 1676 / 1676
Duplicate S1 groups spanning train AND val: 0
[PASS] 4. No duplicate S1 stacks across train/val  --  0 cross-split groups


## Check 5 -- patch id hygiene

Train and val id sets must be disjoint, and every id must resolve to files
that actually exist on disk.

In [7]:
id_overlap = train_set & val_set
missing = []
for pid in paired_ids:
    if not (LIDAR_DIR / f'lidar_patch_{pid}.tif').exists():
        missing.append(('lidar', pid))
    if len(sorted((S1_DIR / f's1_patch_{pid}').glob('t*.tif'))) < 3:
        missing.append(('s1<3views', pid))

print(f'Ids in BOTH train and val: {len(id_overlap)}')
print(f'Ids with missing/incomplete files: {len(missing)}')
if missing:
    print('Examples:', missing[:5])

record('5. Train/val id sets disjoint', 'PASS' if not id_overlap else 'FAIL',
       f'{len(id_overlap)} shared ids')
record('5b. All patch files present', 'PASS' if not missing else 'FAIL',
       f'{len(missing)} incomplete')

Ids in BOTH train and val: 0
Ids with missing/incomplete files: 0
[PASS] 5. Train/val id sets disjoint  --  0 shared ids
[PASS] 5b. All patch files present  --  0 incomplete


## Check 6 -- which Sentinel-1 acquisitions serve which split

Not leakage: the same S1 scene necessarily covers both train and val
patches, since they are drawn from one region. Worth quantifying so the
write-up can state it plainly rather than have an examiner raise it.

In [8]:
def acq_dates(pid, s1_dir):
    p = s1_dir / f's1_patch_{pid}' / 'attrs.json'
    if not p.exists():
        return []
    try:
        return [a.get('acquisition_date') for a in json.load(open(p)) if a.get('acquisition_date')]
    except Exception:
        return []

train_dates = Counter(d for pid in train_ids for d in acq_dates(pid, S1_DIR))
val_dates = Counter(d for pid in val_ids for d in acq_dates(pid, S1_DIR))
shared = set(train_dates) & set(val_dates)

print(f'Distinct S1 acquisition dates -- train: {len(train_dates)}, val: {len(val_dates)}')
print(f'Dates appearing in BOTH: {len(shared)}')
print('Train dates:', sorted(train_dates))
print('Val dates:  ', sorted(val_dates))

record('6. S1 acquisition sharing', 'NOTE',
       f'{len(shared)} dates serve both splits -- expected for a single-region '
       f'study; the split isolates SPACE, not time')

Distinct S1 acquisition dates -- train: 7, val: 7
Dates appearing in BOTH: 7
Train dates: ['2024-03-18', '2024-03-20', '2024-04-01', '2024-04-13', '2024-04-23', '2024-04-25', '2024-05-07']
Val dates:   ['2024-03-18', '2024-03-20', '2024-04-01', '2024-04-13', '2024-04-23', '2024-04-25', '2024-05-07']
[NOTE] 6. S1 acquisition sharing  --  7 dates serve both splits -- expected for a single-region study; the split isolates SPACE, not time


## Check 7 -- unseen-date test really is unseen

`pcrtc/08` does not read its unseen-date imagery from `S1_DIR`; it fetches
new scenes from Planetary Computer at runtime, centred one year after the
LiDAR survey with a +/-30 day window. So the right check is structural:
confirm that no *training* acquisition date falls inside that window. An
earlier version of this cell looked the evaluation patch ids up in the
training directory and compared the training dates against themselves,
which fails unconditionally and means nothing.

In [9]:
import datetime as dt

LIDAR_SURVEY_DATE = dt.date(2024, 4, 16)
UNSEEN_CENTER = LIDAR_SURVEY_DATE.replace(year=LIDAR_SURVEY_DATE.year + 1)  # matches pcrtc/08
SEARCH_DAYS = 30                                                            # matches pcrtc/08
win_lo = UNSEEN_CENTER - dt.timedelta(days=SEARCH_DAYS)
win_hi = UNSEEN_CENTER + dt.timedelta(days=SEARCH_DAYS)

print(f'pcrtc/08 unseen-date search window: {win_lo} to {win_hi}')
parsed = sorted({dt.date.fromisoformat(d) for d in train_dates})
print(f'Training acquisition dates ({len(parsed)}): {parsed[0]} to {parsed[-1]}')

inside = [d for d in parsed if win_lo <= d <= win_hi]
gap_days = min(abs((d - UNSEEN_CENTER).days) for d in parsed)
print(f'Training dates falling INSIDE the unseen window: {len(inside)}')
print(f'Closest training date to the unseen window centre: {gap_days} days')

record('7. Unseen-date window disjoint from training dates',
       'PASS' if not inside else 'FAIL',
       f'closest training acquisition is {gap_days} days from the window centre'
       if not inside else f'{len(inside)} training dates inside the window: {inside}')

Unseen-date evaluation patches: 25
Dates present: ['2024-03-18', '2024-03-20', '2024-04-01', '2024-04-13', '2024-04-23', '2024-04-25', '2024-05-07']
Of those, ALSO in training: ['2024-03-18', '2024-03-20', '2024-04-01', '2024-04-13', '2024-04-23', '2024-04-25', '2024-05-07']
[FAIL] 7. Unseen-date set is date-disjoint from training  --  7 shared dates


## Check 8 -- Cambridge Bay is genuinely a different place

The cross-region claim requires the two regions to be disjoint on the
ground. Compares bounding geometry in WGS84, since the two sets use
different projected CRSs.

In [10]:
if CB_LIDAR_DIR.exists():
    def region_bounds_wgs84(d, limit=400):
        paths = sorted(d.glob('lidar_patch_*.tif'))[:limit]
        L = B = float('inf'); R = T = float('-inf')
        for p in paths:
            with rasterio.open(p) as src:
                l, b, r, t = transform_bounds(src.crs, 'EPSG:4326', *src.bounds)
            L, B, R, T = min(L, l), min(B, b), max(R, r), max(T, t)
        return L, B, R, T

    tuk_b = region_bounds_wgs84(LIDAR_DIR)
    cb_b = region_bounds_wgs84(CB_LIDAR_DIR)
    print('Tuktoyaktuk  (W,S,E,N):', tuple(round(v, 4) for v in tuk_b))
    print('Cambridge Bay(W,S,E,N):', tuple(round(v, 4) for v in cb_b))
    inter = box(*tuk_b).intersection(box(*cb_b))
    sep_deg = box(*tuk_b).distance(box(*cb_b))
    print(f'Bounding-box intersection area: {inter.area:.6f} deg^2')
    print(f'Bounding-box separation: {sep_deg:.3f} deg longitude/latitude')
    id_clash = {p.stem.split('_')[-1] for p in CB_LIDAR_DIR.glob('lidar_patch_*.tif')} & set(paired_ids)
    print(f'Patch ids shared between the two regions: {len(id_clash)} '
          f'(ids are per-region, so a clash is a NAMING collision, not a data one)')
    record('8. Regions geographically disjoint',
           'PASS' if inter.area == 0 else 'FAIL',
           f'separation {sep_deg:.2f} deg')
else:
    record('8. Cross-region check', 'SKIP', 'Cambridge Bay LiDAR directory not found')

Tuktoyaktuk  (W,S,E,N): (-133.3611, 69.7101, -133.3315, 69.8239)
Cambridge Bay(W,S,E,N): (-105.9885, 68.9929, -105.9009, 69.0293)
Bounding-box intersection area: 0.000000 deg^2
Bounding-box separation: 27.352 deg longitude/latitude
Patch ids shared between the two regions: 0 (ids are per-region, so a clash is a NAMING collision, not a data one)
[PASS] 8. Regions geographically disjoint  --  separation 27.35 deg


## Check 9 -- degenerate patches

Near-constant or mostly-masked targets distort correlation metrics: ZNCC on
a flat patch is dominated by noise and can take any value in [-1, 1].

In [11]:
stds = np.array([lidar_stats[p]['std'] for p in paired_ids])
vfrac = np.array([lidar_stats[p]['valid_frac'] for p in paired_ids])

near_flat = stds < 0.01
low_valid = vfrac < 0.5
val_flat = sum(1 for p in val_ids if lidar_stats[p]['std'] < 0.01)

print(f'LiDAR target std: median {np.median(stds):.3f} m, '
      f'5-95% {np.percentile(stds,5):.3f}-{np.percentile(stds,95):.3f}')
print(f'Patches with std < 0.01 m (near-flat): {near_flat.sum()} ({near_flat.mean():.1%})')
print(f'Patches with <50% valid pixels: {low_valid.sum()} ({low_valid.mean():.1%})')
print(f'Near-flat patches inside the VALIDATION set: {val_flat} / {len(val_ids)}')

record('9. Degenerate targets', 'PASS' if val_flat == 0 else 'NOTE',
       f'{val_flat} near-flat validation patches (ZNCC unstable on these)')
record('9b. Mask coverage', 'PASS' if low_valid.sum() == 0 else 'NOTE',
       f'{low_valid.sum()} patches under 50% valid')

LiDAR target std: median 0.158 m, 5-95% 0.068-0.271
Patches with std < 0.01 m (near-flat): 0 (0.0%)
Patches with <50% valid pixels: 0 (0.0%)
Near-flat patches inside the VALIDATION set: 0 / 255
[PASS] 9. Degenerate targets  --  0 near-flat validation patches (ZNCC unstable on these)
[PASS] 9b. Mask coverage  --  0 patches under 50% valid


## Check 10 -- conditioning inputs actually carry variance

The check that would have caught the inert DEM immediately. Any
conditioning input whose within-patch variation is orders of magnitude
below the target's carries no usable signal, however healthy the loss
curves look.

In [12]:
sample = paired_ids[::max(1, len(paired_ids) // 200)]

s1_stds, dem_stds = [], []
for pid in sample:
    times = sorted((S1_DIR / f's1_patch_{pid}').glob('t*.tif'))[:1]
    if times:
        with rasterio.open(times[0]) as src:
            sar = src.read()[:2].astype(np.float32)
        sar = 10.0 * np.log10(np.maximum(np.nan_to_num(sar, nan=1e-12), 1e-12))
        s1_stds.append(float(sar[0].std()))
    dp = DEM_DIR / f'dem_patch_{pid}.tif'
    if dp.exists():
        with rasterio.open(dp) as src:
            dem_stds.append(float(np.nanstd(src.read(1))))

tgt_med = float(np.median(stds))
print(f'Target  (LiDAR)  within-patch std, median: {tgt_med:.4f} m')
if s1_stds:
    print(f'Input   (S1 VV)  within-patch std, median: {np.median(s1_stds):.4f} dB')
if dem_stds:
    dem_med = np.median(dem_stds)
    print(f'Input   (DEM)    within-patch std, median: {dem_med:.4f} m   '
          f'ratio to target: {dem_med / tgt_med:.3f}')

s1_ok = bool(s1_stds) and np.median(s1_stds) > 0.1
record('10. S1 conditioning carries variance', 'PASS' if s1_ok else 'FAIL',
       f'median {np.median(s1_stds):.3f} dB' if s1_stds else 'no S1 read')
if dem_stds:
    dem_ok = (np.median(dem_stds) / tgt_med) > 0.1
    record('10b. DEM conditioning carries variance', 'PASS' if dem_ok else 'FAIL',
           f'ratio {np.median(dem_stds)/tgt_med:.3f} -- CONFIRMED INERT, see CONCEPTS.md'
           if not dem_ok else '')

Target  (LiDAR)  within-patch std, median: 0.1583 m
Input   (S1 VV)  within-patch std, median: 2.2003 dB
Input   (DEM)    within-patch std, median: 0.0038 m   ratio to target: 0.024
[PASS] 10. S1 conditioning carries variance  --  median 2.200 dB
[FAIL] 10b. DEM conditioning carries variance  --  ratio 0.024 -- CONFIRMED INERT, see CONCEPTS.md


## Summary

In [13]:
order = ['FAIL', 'NOTE', 'SKIP', 'PASS']
by_status = defaultdict(list)
for name, (status, detail) in results.items():
    by_status[status].append((name, detail))

print('=' * 72)
for status in order:
    for name, detail in by_status.get(status, []):
        print(f'[{status}] {name}' + (f'  --  {detail}' if detail else ''))
print('=' * 72)

n_fail = len(by_status.get('FAIL', []))
if n_fail == 0:
    print('\nNo contamination found. The NOTE lines are properties to describe')
    print('in the write-up, not defects.')
else:
    print(f'\n{n_fail} FAIL -- resolve before trusting any downstream metric.')

audit_path = OUTPUT_DIR / 's1_pcrtc_dataset_audit.json'
with audit_path.open('w') as h:
    json.dump({k: {'status': v[0], 'detail': v[1]} for k, v in results.items()}, h, indent=2)
print('\nSaved:', audit_path)

[FAIL] 7. Unseen-date set is date-disjoint from training  --  7 shared dates
[FAIL] 10b. DEM conditioning carries variance  --  ratio 0.024 -- CONFIRMED INERT, see CONCEPTS.md
[NOTE] 1. Patch overlap characterised  --  100% of patches overlap a neighbour -- a random split WOULD leak; the block split is load-bearing
[NOTE] 2b. Val patches within buffer distance of train  --  23 val patches lie closer than 150m to a training patch (min 128.0m)
[NOTE] 6. S1 acquisition sharing  --  7 dates serve both splits -- expected for a single-region study; the split isolates SPACE, not time
[PASS] 0. Split reproduces 534/255/887
[PASS] 0b. Single LiDAR CRS  --  EPSG:32608
[PASS] 2. No train/val geometric overlap  --  0 overlapping val patches
[PASS] 3. No duplicate LiDAR across train/val  --  0 cross-split duplicate groups
[PASS] 3b. Duplicate patches within the dataset  --  0 duplicate groups total
[PASS] 4. No duplicate S1 stacks across train/val  --  0 cross-split groups
[PASS] 5. Train/val id se